In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:85% !important; }</style>"))

In [1]:
import os
import glob
import time
import rasterio                  # I/O raster data (netcdf, height, geotiff, ...)
import rasterio.warp             # Reproject raster samples
from rasterio import windows
from rasterio.merge import merge
import fiona                     # I/O vector data (shape, geojson, ...)
import geopandas as gps


from shapely.geometry import Point, Polygon
from shapely.geometry import mapping, shape

import numpy as np               # numerical array manipulation
import os
from tqdm import tqdm
import PIL.Image
import PIL.ImageDraw

from itertools import product
from tensorflow.keras.models import load_model
import cv2

import sys
from core.UNet import UNet
from core.losses import tversky, accuracy, dice_coef, dice_loss, specificity, sensitivity
from core.optimizers import adaDelta, adagrad, adam, nadam
from core.frame_info import FrameInfo, image_normalize
from core.dataset_generator import DataGenerator
from core.split_frames import split_dataset
from core.visualize import display_images

%matplotlib inline
import matplotlib.pyplot as plt  # plotting tools
import matplotlib.patches as patches
from matplotlib.patches import Polygon

import warnings                  # ignore annoying warnings
warnings.filterwarnings("ignore")
import logging
logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)

%reload_ext autoreload
%autoreload 2
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from collections import defaultdict

import tensorflow as tf
print(tf.__version__)

2021-10-28 10:46:00.225182: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0


2.5.0-dev20201225


In [2]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2021-10-28 10:46:06.649369: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcuda.so.1
2021-10-28 10:46:06.693760: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:941] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2021-10-28 10:46:06.696012: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1760] Found device 0 with properties: 
pciBusID: 0000:21:00.0 name: NVIDIA GeForce RTX 3090 computeCapability: 8.6
coreClock: 1.695GHz coreCount: 82 deviceMemorySize: 23.70GiB deviceMemoryBandwidth: 871.81GiB/s
2021-10-28 10:46:06.696056: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0
2021-10-28 10:46:06.712885: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublas.so.11
2021-10-28 10:46:06.712947: I tensorflow/stream_executor/pl

In [3]:
# gpu_options = tf.compat.v1.GPUOptions(per_process_gpu_memory_fraction=0.4)

# sess = tf.compat.v1.Session(config=tf.compat.v1.ConfigProto(gpu_options=gpu_options))

In [27]:
# Required configurations (including the input and output paths) are stored in a separate file (such as config/RasterAnalysis.py)
# Please provide required info in the file before continuing with this notebook. 
 
from config import RasterAnalysis
# In case you are using a different folder name such as configLargeCluster, then you should import from the respective folder 
# Eg. from configLargeCluster import RasterAnalysis

config = RasterAnalysis.Configuration()

In [11]:
# Load a pretrained model
OPTIMIZER = adaDelta
model = load_model(config.trained_model_path, custom_objects={'tversky': tversky, 'dice_coef': dice_coef, 'dice_loss':dice_loss, 'accuracy':accuracy, 'specificity':specificity, 'sensitivity':sensitivity}, compile=False)
model.compile(optimizer=OPTIMIZER, loss=tversky, metrics=[dice_coef, dice_loss, accuracy, specificity, sensitivity])

In [6]:
# Methods to add results of a patch to the total results of a larger area. The operator could be min (useful if there are too many false positives), max (useful for tackle false negatives)
def addTOResult(res, prediction, row, col, he, wi, operator = 'MAX'):
    currValue = res[row:row+he, col:col+wi]
    newPredictions = prediction[:he, :wi]
# IMPORTANT: MIN can't be used as long as the mask is initialed with 0!!!!! If you want to use MIN initial the mask with -1 and handle the case of default value(-1) separately.
    if operator == 'MIN': # Takes the min of current prediction and new prediction for each pixel
        currValue [currValue == -1] = 1 #Replace -1 with 1 in case of MIN
        resultant = np.minimum(currValue, newPredictions) 
    elif operator == 'MAX':
        resultant = np.maximum(currValue, newPredictions)
    else: #operator == 'REPLACE':
        resultant = newPredictions    
# Alternative approach; Lets assume that quality of prediction is better in the centre of the image than on the edges
# We use numbers from 1-5 to denote the quality, where 5 is the best and 1 is the worst.In that case, the best result would be to take into quality of prediction based upon position in account
# So for merge with stride of 0.5, for eg. [12345432100000] AND [00000123454321], should be [1234543454321] instead of [1234543214321] that you will currently get. 
# However, in case the values are strecthed before hand this problem will be minimized
    res[row:row+he, col:col+wi] =  resultant
    return (res)

In [7]:
# Methods that actually makes the predictions
def predict_using_model(model, batch, batch_pos, mask, operator):
    tm = np.stack(batch, axis = 0)
    prediction = model.predict(tm)
    for i in range(len(batch_pos)):
        (col, row, wi, he) = batch_pos[i]
        p = np.squeeze(prediction[i], axis = -1)
        # Instead of replacing the current values with new values, use the user specified operator (MIN,MAX,REPLACE)
        mask = addTOResult(mask, p, row, col, he, wi, operator)
    return mask
    

def detect_tree(img, width=256, height=256, stride = 128, normalize=True):
    # assert ndvi_img.meta['width'] == pan_img.meta['width'] and ndvi_img.meta['height'] == pan_img.meta['height']
    nols, nrows = img.meta['width'], img.meta['height']
    meta = img.meta.copy()
    if 'float' not in meta['dtype']: #The prediction is a float so we keep it as float to be consistent with the prediction. 
        meta['dtype'] = np.float32
    offsets = product(range(0, nols, stride), range(0, nrows, stride))
    big_window = windows.Window(col_off=0, row_off=0, width=nols, height=nrows)
#     print(nrows, nols)

    mask = np.zeros((nrows, nols), dtype=meta['dtype'])

#     mask = mask -1 # Note: The initial mask is initialized with -1 instead of zero to handle the MIN case (see addToResult)
    batch = []
    batch_pos = [ ]
    for col_off, row_off in  tqdm(offsets):
        window =windows.Window(col_off=col_off, row_off=row_off, width=width, height=height).intersection(big_window)
        transform = windows.transform(window, img.transform)
        #patch = np.zeros((height, width, 2)) #Add zero padding in case of corner images
        # number of channels: meta['count']
        patch = np.zeros((height, width, meta['count'])) #Add zero padding in case of corner images
#         ndvi_sm = ndvi_img.read(window=window)
#         pan_sm = pan_img.read(window=window)
#         temp_im = np.stack((ndvi_sm, pan_sm), axis = -1)
#         temp_im = np.squeeze(temp_im)

        #load image with multichannel
        temp_im = img.read(window=window)
        # print('shape shape', temp_im.shape)
        temp_im = np.squeeze(temp_im)
        # channels at the end
        temp_im = np.transpose(temp_im, axes=(1,2,0))
        
        
        
        if normalize:
            temp_im = image_normalize(temp_im, axis=(0,1)) # Normalize the image along the width and height i.e. independently per channel
            
        patch[:window.height, :window.width] = temp_im
        batch.append(patch)
        batch_pos.append((window.col_off, window.row_off, window.width, window.height))
        if (len(batch) == config.BATCH_SIZE):
            mask = predict_using_model(model, batch, batch_pos, mask, 'MAX')
            batch = []
            batch_pos = []
            
    # To handle the edge of images as the image size may not be divisible by n complete batches and few frames on the edge may be left.
    if batch:
        mask = predict_using_model(model, batch, batch_pos, mask, 'MAX')
        batch = []
        batch_pos = []

    return(mask, meta)

In [8]:
schema = {
    'geometry': 'Polygon',
    'properties': {'id': 'str', 'canopy': 'float:15.2',},
    }

def drawPolygons(polygons, shape):
    mask = np.zeros(shape, dtype=np.uint8)
    mask = PIL.Image.fromarray(mask)
    draw = PIL.ImageDraw.Draw(mask)
    for polygon in polygons:
        xy = [(point[1], point[0]) for point in polygon]
        draw.polygon(xy=xy, outline=255, fill=255)
    mask = np.array(mask)#, dtype=bool)   
    return(mask)

def transformToXY(polygons, transform):
    tp = []
    for polygon in polygons:
        rows, cols = zip(*polygon)
        x,y = rasterio.transform.xy(transform, rows, cols)
        tp.append(list(zip(x,y)))
    return (tp)

def createShapefileObject(polygons, meta, wfile):
    with fiona.open(wfile, 'w', crs=meta.get('crs').to_dict(), driver='ESRI Shapefile', schema=schema) as sink:
        for idx, mp in tqdm(enumerate(polygons)):
            try:
#                 poly = Polygon(poly)
    #             assert mp.is_valid
    #             assert mp.geom_type == 'Polygon'
                sink.write({
                    'geometry': mapping(mp),
                    'properties': {'id': str(idx), 'canopy': mp.area},
                })
            except:
                print("An exception occurred in createShapefileObject; Polygon must have more than 2 points")
#                 print(mp)

# Generate a mask with polygons
def transformContoursToXY(contours, transform = None):
    tp = []
    for cnt in contours:
        pl = cnt[:, 0, :]
        cols, rows = zip(*pl)
        x,y = rasterio.transform.xy(transform, rows, cols)
        tl = [list(i) for i in zip(x, y)]
        tp.append(tl)
    return (tp)


def mask_to_polygons(maskF, transform):
    # first, find contours with cv2: it's much faster than shapely
    th = 0.5
    mask = maskF.copy()
    mask[mask < th] = 0
    mask[mask >= th] = 1
    mask = ((mask) * 255).astype(np.uint8)
    contours, hierarchy = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)
    
    #Convert contours from image coordinate to xy coordinate
    contours = transformContoursToXY(contours, transform)
    if not contours: #TODO: Raise an error maybe
        print('Warning: No contours/polygons detected!!')
        return [Polygon()]
    # now messy stuff to associate parent and child contours
    cnt_children = defaultdict(list)
    child_contours = set()
    assert hierarchy.shape[0] == 1
    # http://docs.opencv.org/3.1.0/d9/d8b/tutorial_py_contours_hierarchy.html
    for idx, (_, _, _, parent_idx) in enumerate(hierarchy[0]):
        if parent_idx != -1:
            child_contours.add(idx)
            cnt_children[parent_idx].append(contours[idx])

    # create actual polygons filtering by area (removes artifacts)
    all_polygons = []

    for idx, cnt in enumerate(contours):
        if idx not in child_contours: #and cv2.contourArea(cnt) >= min_area: #Do we need to check for min_area??
            try:
                poly = Polygon(
                    shell=cnt,
                    holes=[c for c in cnt_children.get(idx, [])])
                           #if cv2.contourArea(c) >= min_area]) #Do we need to check for min_area??
                all_polygons.append(poly)
            except:
                pass
#                 print("An exception occurred in createShapefileObject; Polygon must have more than 2 points")
    print(len(all_polygons))
    return(all_polygons)

def create_contours_shapefile(mask, meta, out_fn):
    res = mask_to_polygons(mask, meta['transform'])
#     res = transformToXY(contours, meta['transform'])
    createShapefileObject(res, meta, out_fn)


def writeMaskToDisk(detected_mask, detected_meta, wp, image_type, output_shapefile_type, write_as_type = 'uint8', th = 0.5, create_countors = False):
    # Convert to correct required before writing
    if 'float' in str(detected_meta['dtype']) and 'int' in write_as_type:
        print(f'Converting prediction from {detected_meta["dtype"]} to {write_as_type}, using threshold of {th}')
        detected_mask[detected_mask<th]=0
        detected_mask[detected_mask>=th]=1
        detected_mask = detected_mask.astype(write_as_type)
        detected_meta['dtype'] =  write_as_type
        detected_meta['count'] = 1
        
    with rasterio.open(wp, 'w', **detected_meta) as outds:
        outds.write(detected_mask, 1)
    if create_countors:
        wp = wp.replace(image_type, output_shapefile_type)
        create_contours_shapefile(detected_mask, detected_meta, wp)
        
def mergePredTiff(outputFiles, output_dir, output_prefix, output_image_type, output_shapefile_type, output_dtype,pd):
    """Merge predictions in one single raster tif
    meta: ini meta profile from an individual raster
    outputFiles: list of individual prediction tif files
    """
    src_files_to_mosiac = []

    for k in outputFiles:    
        src = rasterio.open(k)
        src_files_to_mosiac.append(src)
    
    meta = src.meta.copy()
    merged, mergedTrans = merge(src_files_to_mosiac)
    merged = np.squeeze(merged)
    mergedMeta = meta.copy()
    mergedMeta.update({'width':merged.shape[1], 'height': merged.shape[0], 'transform': mergedTrans, 'tiled': True,
            'blockxsize': 256,
            'blockysize': 256,
            'compress': 'lzw',
            'nodata': 0})
    mergedFn = os.path.join(output_dir, f'{output_prefix}merged_{pd}.tif')
    writeMaskToDisk(merged, mergedMeta, mergedFn, image_type = output_image_type, output_shapefile_type = output_shapefile_type, write_as_type = output_dtype, th = 0.5, create_countors = False)            
    print('All predictions merged to a single raster!')
    return

## Year 2019 only

In [ ]:
subfolders = [f.path for f in os.scandir(config.input_image_dir) if f.is_dir()]
print (len(subfolders),subfolders[0])

cutoffFolders = []
for i in subfolders:
    cutoffFolders.append([f.path for f in os.scandir(i) if f.is_dir() and os.path.basename(f)=='cutouts'])
print (len(cutoffFolders),cutoffFolders[0][0].split('/')[-2])
timestr = time.strftime("%Y%m%d-%H%M")
modeldate = os.path.basename(config.trained_model_path).replace('-','_').split('_')[1]


In [ ]:
for i in range(0,len(cutoffFolders)):
    splitFolder = cutoffFolders[i][0]
    gridId = splitFolder.split('/')[-2]
    productDate= f'{modeldate}_{gridId}'
    print (productDate)
    print (splitFolder)
    # Predict trees in the all the files in the input image dir
    # Depending upon the available RAM, images may not to be split before running this cell.
    # Use the Auxiliary-2-SplitRasterToAnalyse if the images are too big to be analysed in memory.
    all_files = glob.glob(f'{splitFolder}/{config.input_image_pref}*.tif',recursive = True)

    print(all_files,len(all_files))
    outputFiles = []
    for j in all_files:
        fullPath = j
        filename = os.path.basename(j)
        gridFolder = os.path.dirname(splitFolder)
        outputFolder = f'{gridFolder}/pred'
        print (outputFolder)
        if not os.path.exists(outputFolder):
            os.makedirs(outputFolder)
        outputFile = os.path.join(outputFolder, config.output_prefix+filename)
        outputFiles.append(outputFile)
        if not os.path.isfile(outputFile) or config.overwrite_analysed_files:               
            with rasterio.open(fullPath) as img:
                print(fullPath)
                detectedMask, detectedMeta = detect_tree(img, width = config.WIDTH, height = config.HEIGHT, stride = config.STRIDE) # WIDTH and HEIGHT should be the same and in this case Stride is 50 % width
                #Write the mask to file
                writeMaskToDisk(detectedMask, detectedMeta, outputFile, image_type = config.output_image_type, output_shapefile_type = config.output_shapefile_type, write_as_type = config.output_dtype, th = 0.5, create_countors = False)            
        else:
            print('File already analysed!', fullPath)
    # merge multiple prediction tifs
    if not os.path.isfile(f'{gridFolder}/{config.output_prefix}_merged_{productDate}.tif'):
        mergePredTiff(outputFiles, gridFolder, config.output_prefix, config.output_image_type, config.output_shapefile_type, config.output_dtype,productDate)
    else:
        print ('Files already merged!')

## Other years

In [28]:
subfolders = [f.path for f in os.scandir(config.input_image_dir) if f.is_dir()]

print (len(subfolders),subfolders[0])
subfolders.sort()
cutoffFolders = subfolders#[:-1]
print (len(cutoffFolders),cutoffFolders[0])
timestr = time.strftime("%Y%m%d-%H%M") 
modeldate = os.path.basename(config.trained_model_path).replace('-','_').split('_')[1] 
productDate= f'{modeldate}_{timestr}'
print (productDate)

72 /mnt/warehouse/sahel/mali/2016/00013_-0010
72 /mnt/warehouse/sahel/mali/2016/00010_-0001
20211010_20211104-0902


In [29]:
for i in range(0,len(cutoffFolders)):
    splitFolder = cutoffFolders[i]
    gridId = splitFolder.split('/')[-1]
    productDate= f'{modeldate}_{gridId}'
    print (productDate)
    print (splitFolder)
    # Predict trees in the all the files in the input image dir
    # Depending upon the available RAM, images may not to be split before running this cell.
    # Use the Auxiliary-2-SplitRasterToAnalyse if the images are too big to be analysed in memory.
    all_files = glob.glob(f'{splitFolder}/{config.input_image_pref}*.tif',recursive = True)

    print(all_files[0],len(all_files))
    outputFiles = []
    for j in all_files:
        fullPath = j
        filename = os.path.basename(j)
        gridFolder = splitFolder
        outputFolder = f'{gridFolder}/pred'
        print (outputFolder)
        if not os.path.exists(outputFolder):
            os.makedirs(outputFolder)
        outputFile = os.path.join(outputFolder, config.output_prefix+filename)
        outputFiles.append(outputFile)
        if not os.path.isfile(outputFile) or config.overwrite_analysed_files:               
            with rasterio.open(fullPath) as img:
                print(fullPath)
                detectedMask, detectedMeta = detect_tree(img, width = config.WIDTH, height = config.HEIGHT, stride = config.STRIDE) # WIDTH and HEIGHT should be the same and in this case Stride is 50 % width
                #Write the mask to file
                writeMaskToDisk(detectedMask, detectedMeta, outputFile, image_type = config.output_image_type, output_shapefile_type = config.output_shapefile_type, write_as_type = config.output_dtype, th = 0.5, create_countors = False)            
        else:
            print('File already analysed!', fullPath)
    # merge multiple prediction tifs
    if not os.path.isfile(f'{gridFolder}/{config.output_prefix}merged_{productDate}.tif'):
        mergePredTiff(outputFiles, gridFolder, config.output_prefix, config.output_image_type, config.output_shapefile_type, config.output_dtype,productDate)
    else:
        print ('Files already merged!')

20211010_00010_-0001
/mnt/warehouse/sahel/mali/2016/00010_-0001
/mnt/warehouse/sahel/mali/2016/00010_-0001/L15-1022E-1082N_wgs84.tif 42
/mnt/warehouse/sahel/mali/2016/00010_-0001/pred


0it [00:00, ?it/s]

/mnt/warehouse/sahel/mali/2016/00010_-0001/L15-1022E-1082N_wgs84.tif


272it [00:06, 39.20it/s] 
0it [00:00, ?it/s]

Converting prediction from <class 'numpy.float32'> to uint8, using threshold of 0.5
/mnt/warehouse/sahel/mali/2016/00010_-0001/pred
/mnt/warehouse/sahel/mali/2016/00010_-0001/L15-1022E-1086N_wgs84.tif


...

Converting prediction from <class 'numpy.float32'> to uint8, using threshold of 0.5
/mnt/warehouse/sahel/mali/2016/00015_-0012/pred
/mnt/warehouse/sahel/mali/2016/00015_-0012/L15-0961E-1112N_wgs84.tif


272it [00:09, 29.89it/s] 
0it [00:00, ?it/s]

Converting prediction from <class 'numpy.float32'> to uint8, using threshold of 0.5
/mnt/warehouse/sahel/mali/2016/00015_-0012/pred
/mnt/warehouse/sahel/mali/2016/00015_-0012/L15-0957E-1111N_wgs84.tif


272it [00:08, 33.07it/s] 


Converting prediction from <class 'numpy.float32'> to uint8, using threshold of 0.5
All predictions merged to a single raster!


In [ ]:
f'{gridFolder}/{config.output_prefix}_merged_{productDate}.tif'

In [ ]:
import os
import glob
from osgeo import gdal
out_dir = '/mnt/warehouse/planet_medres_normalized_analytic_2019-06_2019-11_mosaic/result'
fileList = glob.glob(f'{out_dir}/*.tif')
# fileList
# with open('/mnt/warehouse/planet_medres_normalized_analytic_2019-06_2019-11_mosaic/result/listfile.txt', 'w') as filehandle:
#     for listitem in fileList:
#         filehandle.write('%s\n' % listitem)

In [ ]:
warp_options = dict(
    multithread=True,
    dstSRS="EPSG:4326",
    warpMemoryLimit=32000000000,
    creationOptions=["TILED=YES", "BIGTIFF=IF_SAFER", "NUM_THREADS=ALL_CPUS","ALPHA=NO","COMPRESSION=LZW"],
    warpOptions=[f"NUM_THREADS=ALL_CPUS"],
#         callback=gdalwarp_progress_callback,
#         callback_data=tqdm(total=100, desc=f"Merging mosaic for  {folder}", position=2, leave=False)
)
merged_fp = f"/{out_dir}/nicif_mosaic_crop_merge_2019.tif"
gdal.Warp(merged_fp, fileList, **warp_options)

## Predict trees in the all the files in the input image dir
# Depending upon the available RAM, images may not to be split before running this cell.
# Use the Auxiliary-2-SplitRasterToAnalyse if the images are too big to be analysed in memory.
all_files = []
subfolders = [f.path for f in os.scandir(config.input_image_dir) if f.is_dir()]
for i in subfolders:
    
for root, dirs, files in os.walk(config.input_image_dir):
    for file in files:
        if file.endswith(config.input_image_type):# and file.startswith(config.ndvi_fn_st):
             all_files.append((os.path.join(root, file), file))
print(len(all_files))


In [ ]:
for fullPath, filename in all_files:
    outputFile = os.path.join(config.output_dir, filename.replace(config.ndvi_fn_st, config.output_prefix) )
#     outputFile = os.path.join(config.outputTime, config.output_prefix+filename)
    if not os.path.isfile(outputFile) or config.overwrite_analysed_files: 
#         with rasterio.open(fullPath) as ndvi:
#             with rasterio.open(fullPath.replace(config.ndvi_fn_st, config.pan_fn_st)) as pan:
#                 print(fullPath)
#                 detectedMask, detectedMeta = detect_tree(ndvi, pan, width = config.WIDTH, height = config.HEIGHT, stride = config.STRIDE) # WIDTH and HEIGHT should be the same and in this case Stride is 50 % width
#                 #Write the mask to file
#                 writeMaskToDisk(detectedMask, detectedMeta, outputFile, write_as_type = config.output_dtype, th = 0.5, create_countors = False)                
        with rasterio.open(fullPath) as img:
            print(fullPath)
            detectedMask, detectedMeta = detect_tree(img, width = config.WIDTH, height = config.HEIGHT, stride = config.STRIDE) # WIDTH and HEIGHT should be the same and in this case Stride is 50 % width
            #Write the mask to file
            writeMaskToDisk(detectedMask, detectedMeta, outputFile, image_type = config.output_image_type, output_shapefile_type = config.output_shapefile_type, write_as_type = config.output_dtype, th = 0.5, create_countors = False)            

    else:
        print('File already analysed!', fullPath)

In [ ]:
# # Display extracted image
# sampleImage = 'ps_PSScene4Band_2019_00013_00062_198_257_composite.tif'
# fn = os.path.join(config.output_dir, config.output_prefix + sampleImage )
# predicted_img = rasterio.open(fn)
# p = predicted_img.read()
# np.unique(p, return_counts=True)
# plt.imshow(p[0])

In [ ]:
# merge predictions
import os
import numpy as np
import time
import glob
import shutil
import rasterio
from rasterio import features as rfeatures
from rasterio.merge import merge
from rasterio.mask import mask

In [ ]:
j = f'/media/xiaoye/ps/fallow/result_crop_20201112-1652'
quadsFile = glob.glob(f'{j}/*.tif',recursive = True)
print (len(quadsFile))

In [ ]:

wgsFile = glob.glob(f'{j}/*.tif',recursive = True)
print ('prediction: ', len(quadsFile))

fileSpace = np.linspace(0,13000,14)
fileSpace =np.append(fileSpace,13339)

for i in range(0,len(fileSpace)-1):
    print (i)
    src_files_to_mosiac = []
    startP = int(fileSpace[i])
    endP = int(fileSpace[i+1])
    print (startP, endP)
    for k in wgsFile[startP:endP]:    
        src = rasterio.open(k)
        src_files_to_mosiac.append(src)

    mosaic, out_trans = merge(src_files_to_mosiac)

    mosaicfn = f'{j}/nicif_{os.path.basename(j)}_{i}.tif'

    src = rasterio.open(wgsFile[i])
    outMeta = src.meta.copy()
    outMeta.update({"driver":"GTiff","height": mosaic.shape[1],"width": mosaic.shape[2],"transform": out_trans,'dtype': 'uint8', 'interleave': 'band',
        'tiled': True,
        'blockxsize': 256,
        'blockysize': 256,
        'compress': 'lzw'})
    with rasterio.open(mosaicfn, "w", **outMeta, num_threads='all_cpus') as dest:
        dest.write(mosaic)
#     #remove the wgs84 files
#     for x in wgsFile[startP:endP]:
#         os.remove(x)

In [ ]:
fileSpace = np.linspace(1000,13000,13)
fileSpace =np.append(fileSpace,13340)
fileSpace

In [ ]:
wgsFile[startP:endP]